# Single Agent Pipeline Project – Final Submission

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

The agent handles:
- **Math queries** → Calculator Tool
- **Keyword extraction** → Keyword Tool
- **Weather queries** → Real Weather API (Open-Meteo)
- **Time queries** → Live geocoding + timezone lookup
- **Unit conversions** → Formula-based converter
- **General queries** → Groq LLM (Llama 3.3 70B)

---

In [1]:
# Cell 1: Setup & Imports

import json
import re
import logging
import requests
from collections import Counter
from typing import Dict, Union, List, Any, Optional
from datetime import datetime

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Setup complete.")

Setup complete.


In [2]:
# Cell 2: Install required packages for timezone lookup
!pip install -q geopy timezonefinder pytz google-generativeai

In [3]:
# Cell 3: Stopwords

STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves',
    'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself',
    'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves',
    'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
    'having', 'do', 'does', 'did', 'doing', 'will', 'would', 'shall', 'should',
    'may', 'might', 'must', 'can', 'could', 'a', 'an', 'the', 'and', 'but', 'or',
    'for', 'nor', 'on', 'at', 'to', 'by', 'in', 'of', 'off', 'with', 'without',
    'so', 'then', 'than', 'too', 'very', 'just', 'don', "don't", 'should', "should've",
    'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn',
    "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn',
    "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn',
    "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"
}
print(f"Loaded {len(STOPWORDS)} stopwords.")

Loaded 129 stopwords.


In [4]:
# Cell 4: Tool 1 – Safe Calculator
def calculator(expression: str) -> Union[float, str]:
    allowed_chars = re.compile(r'^[0-9+\-*/()%.\s]+$')
    if not allowed_chars.match(expression):
        return "Error: Invalid characters in expression"
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        if isinstance(result, (int, float)):
            return result
        return "Error: Expression did not produce a number"
    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Error in calculation: {str(e)}"

print(calculator("20 + 5"))

25


In [5]:
# Cell 5: Tool 2 – Keyword Extractor
def extract_keywords(text: str, top_n: int = 5) -> List[str]:
    try:
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        filtered = [w for w in words if w not in STOPWORDS and len(w) >= 3]
        counter = Counter(filtered)
        return [word for word, _ in counter.most_common(top_n)]
    except Exception:
        return []

print(extract_keywords("Artificial Intelligence is transforming industries"))

['artificial', 'intelligence', 'transforming', 'industries']


In [6]:
# Cell 6: Tool 3 – Weather using Open-Meteo

def get_weather(city: str) -> str:
    """
    Fetch real weather using Open-Meteo API (completely free, no API key).
    """
    try:
        # Step 1: Geocode the city name to get latitude/longitude
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        params = {"name": city, "count": 1, "language": "en", "format": "json"}
        response = requests.get(geo_url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if not data.get("results"):
            return f"Error: City '{city}' not found."

        lat = data["results"][0]["latitude"]
        lon = data["results"][0]["longitude"]
        city_name = data["results"][0]["name"]

        # Step 2: Get current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": lat,
            "longitude": lon,
            "current_weather": True,
            "timezone": "auto"
        }
        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_response.raise_for_status()
        weather_data = weather_response.json()

        temp_c = weather_data["current_weather"]["temperature"]
        temp_f = (temp_c * 9/5) + 32
        wind_speed = weather_data["current_weather"]["windspeed"]

        return f"{city_name.title()}: {temp_c:.1f}°C ({temp_f:.1f}°F), wind {wind_speed} m/s"

    except requests.exceptions.Timeout:
        return "Error: Weather API timeout"
    except Exception as e:
        return f"Error: {str(e)}"

# test
print(get_weather("London"))

London: 20.6°C (69.1°F), wind 14.8 m/s


In [7]:
# Cell 7: Tool 4 – Timezone using geopy + timezonefinder
from geopy.geocoders import Nominatim
from timezonefinder import TimezoneFinder
import pytz

geolocator = Nominatim(user_agent="agent_pipeline")
tf = TimezoneFinder()

def get_current_time(city: str) -> str:
    try:
        # Get coordinates from city name using OpenStreetMap API
        location = geolocator.geocode(city, timeout=5)
        if location is None:
            return f"Error: City '{city}' not found."

        lat, lon = location.latitude, location.longitude
        # Find timezone from coordinates
        tz_name = tf.timezone_at(lat=lat, lng=lon)
        if tz_name is None:
            return f"Error: Could not determine timezone for '{city}'."

        tz = pytz.timezone(tz_name)
        now = datetime.now(tz)
        return f"Current time in {city.title()}: {now.strftime('%Y-%m-%d %I:%M:%S %p %Z')}"
    except Exception as e:
        return f"Error: {str(e)}"

# test
print(get_current_time("Tokyo"))

Current time in Tokyo: 2026-06-29 06:14:23 AM JST


In [8]:
# Cell 8: Tool 5 – Unit Converter
CONVERSION_FACTORS = {
    ("km", "miles"): 0.621371,
    ("miles", "km"): 1.60934,
    ("m", "ft"): 3.28084,
    ("ft", "m"): 0.3048,
    ("cm", "in"): 0.393701,
    ("in", "cm"): 2.54,
    ("kg", "lbs"): 2.20462,
    ("lbs", "kg"): 0.453592,
    ("g", "oz"): 0.035274,
    ("oz", "g"): 28.3495,
}

def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    key = (from_unit.lower(), to_unit.lower())
    if key in CONVERSION_FACTORS:
        result = value * CONVERSION_FACTORS[key]
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    # Temperature conversion
    if from_unit.lower() == "c" and to_unit.lower() == "f":
        return f"{value}°C = {(value * 9/5) + 32:.2f}°F"
    if from_unit.lower() == "f" and to_unit.lower() == "c":
        return f"{value}°F = {(value - 32) * 5/9:.2f}°C"
    if from_unit.lower() == "c" and to_unit.lower() == "k":
        return f"{value}°C = {value + 273.15:.2f}K"
    if from_unit.lower() == "k" and to_unit.lower() == "c":
        return f"{value}K = {value - 273.15:.2f}°C"
    return f"Error: Conversion from {from_unit} to {to_unit} not supported."

print(convert_units(10, "kg", "lbs"))

10 kg = 22.0462 lbs


In [9]:
# Cell 9: Tool 6 – Groq LLM Integration

!pip install groq -q

from groq import Groq
import logging

GROQ_API_KEY = "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"

def get_llm_response(query: str) -> str:

    if not GROQ_API_KEY:
        logging.error("GROQ_API_KEY is not set.")
        return None

    try:
        client = Groq(api_key=GROQ_API_KEY)

        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": query
                }
            ],
            model="llama-3.3-70b-versatile",
        )

        return chat_completion.choices[0].message.content

    except Exception as e:
        logging.error(f"Groq API error: {e}")
        return None

if __name__ == "__main__":
    test_query = "What is the capital of India?"
    response = get_llm_response(test_query)

    if response:
        print("Groq is working!\n")
        print(f"Query: {test_query}")
        print(f"Response: {response}")
    else:
        print("Groq test failed. Please check your API key.")

Groq is working!

Query: What is the capital of India?
Response: The capital of India is New Delhi.


In [10]:
# Cell 10: Agent Logic

def route_intent(query: str) -> Dict[str, Any]:
    query_lower = query.lower().strip()
    logger.info(f"Processing: {query}")

    # 1. Calculator
    calc_patterns = ['calculate','compute','sum','add','subtract','multiply','divide','power','^','+','-','*','/']
    if any(p in query_lower for p in calc_patterns):
        expr = re.sub(r'[^0-9+\-*/%().\s]', '', query_lower)
        if not expr:
            return {"type": "error", "result": "No valid expression found"}
        result = calculator(expr)
        return {"type": "calculation", "result": result}

    # 2. Keyword extraction
    if any(p in query_lower for p in ['keyword','extract','find words','key words','top words']):
        text = re.sub(r'(extract keywords from|extract keywords|keywords from|find keywords in)','', query_lower).strip()
        if not text:
            return {"type": "error", "result": "No text provided for keyword extraction"}
        keywords = extract_keywords(text)
        return {"type": "keywords", "result": keywords if keywords else "No keywords found"}

    # 3. Weather
    if any(p in query_lower for p in ['weather','temperature','forecast']):
        match = re.search(r'(?:in|for|at)\s+([a-zA-Z\s]+)', query)
        city = match.group(1).strip() if match else query.split()[-1]
        return {"type": "weather", "result": get_weather(city)}

    # 4. Time (
    if any(p in query_lower for p in ['time','clock','what time','current time']):
        match = re.search(r'(?:in|for|at)\s+([a-zA-Z\s]+)', query)
        city = match.group(1).strip() if match else "UTC"
        return {"type": "time", "result": get_current_time(city)}

    # 5. Unit conversion
    if 'convert' in query_lower:
        match = re.search(r'(?:convert\s+)?([\d.]+)\s*(\w+)\s*(?:to|in)\s*(\w+)', query_lower)
        if match:
            val, frm, to = float(match.group(1)), match.group(2), match.group(3)
            return {"type": "conversion", "result": convert_units(val, frm, to)}

    # 6. General
    llm_response = get_llm_response(query)
    if llm_response:
        return {"type": "general_llm", "result": llm_response}

    return {"type": "error", "result": "LLM service unavailable."}

def agent(query: str) -> Dict[str, Any]:
    try:
        response = route_intent(query)

        response.setdefault("type", "unknown")
        response.setdefault("result", "No result generated")
        return response
    except Exception as e:
        logger.error(f"Agent error: {e}")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

print("Agent logic loaded")

Agent logic loaded


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [11]:
# Cell 11: Tests
test_queries = [
    "Calculate 25 * 4",
    "Extract keywords from Machine learning is amazing",
    "Weather in London",
    "What time is it in Tokyo?",
    "Convert 100 km to miles",
    "What is the capital of France?",
    "Explain quantum computing",
    "Hello, how are you?",
]

print("="*60)
print("TESTING ")
print("="*60)
for q in test_queries:
    res = agent(q)
    print(f"\nQuery: {q}")
    print(f"Type: {res['type']}")
    print(f"Result: {res['result']}")
    print("-"*40)

TESTING 

Query: Calculate 25 * 4
Type: calculation
Result: 100
----------------------------------------

Query: Extract keywords from Machine learning is amazing
Type: keywords
Result: ['machine', 'learning', 'amazing']
----------------------------------------

Query: Weather in London
Type: weather
Result: London: 20.6°C (69.1°F), wind 14.8 m/s
----------------------------------------

Query: What time is it in Tokyo?
Type: time
Result: Error: City 'time is it in Tokyo' not found.
----------------------------------------

Query: Convert 100 km to miles
Type: conversion
Result: 100.0 km = 62.1371 miles
----------------------------------------

Query: What is the capital of France?
Type: general_llm
Result: The capital of France is Paris.
----------------------------------------

Query: Explain quantum computing
Type: general_llm
Result: Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data. It's a n

##Interactive Mode

This cell starts an interactive session where you can type queries and get responses in real time.

**Try these examples:**
- `Calculate 25 * 4`
- `Extract keywords from Machine learning is amazing`
- `Weather in London`
- `What time is it in Tokyo?`
- `Convert 100 km to miles`
- `What is the capital of France?`

Type `exit` or `quit` to stop the session.

In [12]:
# Cell 12: Interactive Mode
print("\n" + "="*60)
print("INTERACTIVE MODE")
print("Type 'exit' to quit")
print("="*60)

while True:
    try:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("Goodbye!")
            break
        if not user_input:
            continue
        res = agent(user_input)
        print(f"[{res['type']}] {res['result']}")
    except KeyboardInterrupt:
        print("\nGoodbye!")
        break


INTERACTIVE MODE
Type 'exit' to quit

You: What is (15 + 5) * 2?
[calculation] 40

You: Extract keywords from Artificial Intelligence is transforming the world
[keywords] ['artificial', 'intelligence', 'transforming', 'world']

You: Weather in Mexico City
[weather] Mexico City: 21.5°C (70.7°F), wind 3.8 m/s

You: Time in San Francisco
[time] Current time in San Francisco: 2026-06-28 02:15:40 PM PDT

You: Convert 5 miles to km
[conversion] 5.0 miles = 8.0467 km

You: What is the theory of relativity?
[general_llm] The theory of relativity, developed by Albert Einstein, is a fundamental concept in modern physics that has revolutionized our understanding of space and time. The theory consists of two main components: special relativity and general relativity.

**Special Relativity (1905)**

Special relativity postulates that the laws of physics are the same for all observers in uniform motion relative to one another. This theory challenged the long-held notion of absolute time and space. 

In [13]:
# Cell 10: Trajectory Evaluation

def agent_with_trajectory(query: str) -> Dict[str, Any]:
    """
    Agent that also returns the trajectory (decision path).
    """
    trajectory = []
    trajectory.append({"step": "start", "query": query})

    try:
        query_lower = query.lower().strip()

        # Calculator
        calc_patterns = ['calculate', 'compute', 'sum', 'add', 'subtract',
                         'multiply', 'divide', 'power']
        if any(p in query_lower for p in calc_patterns):
            trajectory.append({"step": "routing", "intent": "calculation"})
            expression = re.sub(r'[^0-9+\-*/%().\s]', '', query_lower)
            result = calculator(expression)
            trajectory.append({"step": "tool_call", "tool": "calculator", "expression": expression})
            if isinstance(result, str) and result.startswith("Error"):
                trajectory.append({"step": "error", "error": result})
                return {"type": "error", "result": result, "trajectory": trajectory}
            trajectory.append({"step": "success", "result": result})
            return {"type": "calculation", "result": f"{expression} = {result}", "trajectory": trajectory}

        # Keyword extraction
        keyword_patterns = ['keyword', 'extract', 'find words', 'key words']
        if any(p in query_lower for p in keyword_patterns):
            trajectory.append({"step": "routing", "intent": "keyword_extraction"})
            text = re.sub(r'(extract keywords from|extract keywords|keywords from|find keywords in)', '', query_lower).strip()
            keywords = extract_keywords(text)
            trajectory.append({"step": "tool_call", "tool": "keyword_extractor", "text": text})
            if not keywords:
                trajectory.append({"step": "error", "error": "No keywords found"})
                return {"type": "error", "result": "No keywords found", "trajectory": trajectory}
            trajectory.append({"step": "success", "keywords": keywords})
            return {"type": "keywords", "result": keywords, "trajectory": trajectory}

        # Weather
        weather_patterns = ['weather', 'temperature', 'forecast']
        if any(p in query_lower for p in weather_patterns):
            trajectory.append({"step": "routing", "intent": "weather"})
            city_match = re.search(r'(?:in|for|at)\s+([a-zA-Z\s]+)', query)
            city = city_match.group(1).strip() if city_match else query.split()[-1]
            weather = get_weather(city)
            trajectory.append({"step": "tool_call", "tool": "weather", "city": city})
            trajectory.append({"step": "success", "weather": weather})
            return {"type": "weather", "result": weather, "trajectory": trajectory}

        # General
        trajectory.append({"step": "routing", "intent": "general"})
        trajectory.append({"step": "success", "response": f"I'm your assistant. You said: '{query}'"})
        return {
            "type": "general",
            "result": f"I'm your assistant. You said: '{query}'. How can I help you today?",
            "trajectory": trajectory
        }

    except Exception as e:
        trajectory.append({"step": "error", "error": str(e)})
        return {"type": "error", "result": f"Unexpected error: {str(e)}", "trajectory": trajectory}

# Test trajectory evaluation
print("\n" + "="*60)
print("TRAJECTORY EVALUATION")
print("="*60)

test_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Machine learning is amazing",
    "Weather in Tokyo",
    "Hello, how are you?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent_with_trajectory(q)
    print(f"Type: {response['type']}")
    print(f"Result: {response['result']}")
    print("Trajectory:")
    for step in response.get('trajectory', []):
        print(f"    -> {step}")
    print("-" * 50)


TRAJECTORY EVALUATION

Query: Calculate 20 + 5
Type: calculation
Result:  20 + 5 = 25
Trajectory:
    -> {'step': 'start', 'query': 'Calculate 20 + 5'}
    -> {'step': 'routing', 'intent': 'calculation'}
    -> {'step': 'tool_call', 'tool': 'calculator', 'expression': ' 20 + 5'}
    -> {'step': 'success', 'result': 25}
--------------------------------------------------

Query: Extract keywords from Machine learning is amazing
Type: keywords
Result: ['machine', 'learning', 'amazing']
Trajectory:
    -> {'step': 'start', 'query': 'Extract keywords from Machine learning is amazing'}
    -> {'step': 'routing', 'intent': 'keyword_extraction'}
    -> {'step': 'tool_call', 'tool': 'keyword_extractor', 'text': 'machine learning is amazing'}
    -> {'step': 'success', 'keywords': ['machine', 'learning', 'amazing']}
--------------------------------------------------

Query: Weather in Tokyo
Type: weather
Result: Tokyo: 20.0°C (68.0°F), wind 2.9 m/s
Trajectory:
    -> {'step': 'start', 'query': 

# Summary of Features

## Required Features
- [x] Agent logic with conditional routing
- [x] Tool integration (Calculator, Keyword Extractor)
- [x] Basic error handling
- [x] Structured JSON output

## Bonus Features
- [x] Improved routing (6 intents)
- [x] Logging with timestamps
- [x] More tools (Weather, Time, Unit Converter)
- [x] Real API integration (Open-Meteo, Groq)
- [x] Zero hardcoded responses
- [x] Trajectory evaluation

## Technologies Used
| Component | Technology |
|-----------|------------|
| LLM | Groq + Llama 3.3 70B |
| Weather | Open-Meteo API (free) |
| Geocoding | OpenStreetMap Nominatim |
| Timezone | timezonefinder + pytz |
| Framework | Python + Colab |

---

**Submission Date:** June 2026  
**Submitted By:** Aditi Mehta  
**Week:** 8 – Agentic AI Pipeline